# CAN Bus Intrusion Detection System
## Deep Learning + Voltage Fingerprinting

This notebook demonstrates the complete CAN IDS pipeline:
1. Data loading and preprocessing
2. Voltage fingerprinting
3. Deep learning models (CNN & LSTM)
4. Decision-level fusion
5. Evaluation and comparison

In [ ]:
# Setup
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.append('../src')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Setup complete!")

## 1. Data Loading and Exploration

In [ ]:
from dataset_loader import CANDatasetLoader

# Initialize loader
loader = CANDatasetLoader("../data/raw")

# Load datasets
print("Loading voltage dataset...")
voltage_df = loader.load_canmap_voltage_dataset()
print(f"Voltage dataset shape: {voltage_df.shape}")
print("\nVoltage dataset sample:")
display(voltage_df.head())

print("\nLoading CAN message dataset...")
can_df = loader.load_road_dataset()
print(f"CAN dataset shape: {can_df.shape}")
print("\nCAN dataset sample:")
display(can_df.head())

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Voltage dataset
voltage_df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Voltage Dataset - Label Distribution')
axes[0].set_xlabel('Label (0=Normal, 1=Attack)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Normal', 'Attack'], rotation=0)

# CAN dataset
can_df['label'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('CAN Message Dataset - Label Distribution')
axes[1].set_xlabel('Label (0=Normal, 1=Attack)')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Normal', 'Attack'], rotation=0)

plt.tight_layout()
plt.show()

## 2. Data Preprocessing

In [ ]:
# Preprocess voltage data
print("Preprocessing voltage data...")
X_voltage, y_voltage = loader.preprocess_voltage_data(voltage_df)
print(f"Voltage features shape: {X_voltage.shape}")
print(f"Voltage labels shape: {y_voltage.shape}")

# Preprocess CAN message data
print("\nPreprocessing CAN message data...")
X_can, y_can = loader.preprocess_can_data(can_df, sequence_length=100)
print(f"CAN sequences shape: {X_can.shape}")
print(f"CAN labels shape: {y_can.shape}")

In [ ]:
# Split data
voltage_splits = loader.split_data(X_voltage, y_voltage)
can_splits = loader.split_data(X_can, y_can)

print("Voltage splits:")
for split_name, (X, y) in voltage_splits.items():
    print(f"  {split_name}: X={X.shape}, y={y.shape}")

print("\nCAN splits:")
for split_name, (X, y) in can_splits.items():
    print(f"  {split_name}: X={X.shape}, y={y.shape}")

## 3. Voltage Fingerprinting

In [ ]:
from voltage_fingerprinting import VoltageFingerprinter

# Get training data
X_train_v, y_train_v = voltage_splits['train']
X_test_v, y_test_v = voltage_splits['test']

# Extract ECU IDs
ecu_ids_train = voltage_df['ecu_id'].values[:len(X_train_v)]

# Train fingerprinter
print("Training voltage fingerprinter...")
fingerprinter = VoltageFingerprinter(threshold=0.7)
fingerprinter.train(X_train_v, ecu_ids_train)

print(f"\nTrained on {len(fingerprinter.ecu_profiles)} ECUs")
print(f"ECU IDs: {list(fingerprinter.ecu_profiles.keys())}")

In [ ]:
# Test voltage fingerprinting
voltage_predictions = []
voltage_scores = []

for i in range(len(X_test_v)):
    claimed_ecu = voltage_df['ecu_id'].values[len(X_train_v) + i]
    is_anomaly, confidence = fingerprinter.detect_anomaly(X_test_v[i], claimed_ecu)
    voltage_predictions.append(int(is_anomaly))
    voltage_scores.append(1.0 - confidence)

voltage_predictions = np.array(voltage_predictions)
voltage_scores = np.array(voltage_scores)

print(f"Predictions made: {len(voltage_predictions)}")
print(f"Anomalies detected: {np.sum(voltage_predictions)}")
print(f"Actual attacks: {np.sum(y_test_v)}")

## 4. Deep Learning Models

In [ ]:
from deep_learning_models import CNNModel, LSTMModel

# Get training data
X_train_can, y_train_can = can_splits['train']
X_val_can, y_val_can = can_splits['val']
X_test_can, y_test_can = can_splits['test']

input_shape = X_train_can.shape[1:]  # (sequence_length, features)
print(f"Input shape: {input_shape}")

In [ ]:
# Train CNN
print("Training CNN model...")
cnn = CNNModel(filters=[64, 128, 256], dropout=0.3)
cnn.build_model(input_shape=input_shape)
cnn.model.summary()

# Train
cnn_history = cnn.train(
    X_train_can, y_train_can,
    X_val_can, y_val_can,
    epochs=20,  # Reduced for demo
    batch_size=32,
    learning_rate=0.001
)

In [ ]:
# Train LSTM
print("Training LSTM model...")
lstm = LSTMModel(hidden_units=[128, 64], bidirectional=True, dropout=0.2)
lstm.build_model(input_shape=input_shape)
lstm.model.summary()

# Train
lstm_history = lstm.train(
    X_train_can, y_train_can,
    X_val_can, y_val_can,
    epochs=20,  # Reduced for demo
    batch_size=32,
    learning_rate=0.001
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# CNN Loss
axes[0, 0].plot(cnn_history['loss'], label='Train Loss')
axes[0, 0].plot(cnn_history['val_loss'], label='Val Loss')
axes[0, 0].set_title('CNN - Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# CNN Accuracy
axes[0, 1].plot(cnn_history['accuracy'], label='Train Accuracy')
axes[0, 1].plot(cnn_history['val_accuracy'], label='Val Accuracy')
axes[0, 1].set_title('CNN - Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# LSTM Loss
axes[1, 0].plot(lstm_history['loss'], label='Train Loss')
axes[1, 0].plot(lstm_history['val_loss'], label='Val Loss')
axes[1, 0].set_title('LSTM - Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

# LSTM Accuracy
axes[1, 1].plot(lstm_history['accuracy'], label='Train Accuracy')
axes[1, 1].plot(lstm_history['val_accuracy'], label='Val Accuracy')
axes[1, 1].set_title('LSTM - Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
from evaluation_metrics import IDSEvaluator

evaluator = IDSEvaluator(save_dir="../results/notebook_demo")

# Get CNN predictions
cnn_predictions, cnn_scores = cnn.predict(X_test_can)

# Get LSTM predictions
lstm_predictions, lstm_scores = lstm.predict(X_test_can)

# Calculate metrics
voltage_metrics = evaluator.calculate_metrics(y_test_v, voltage_predictions, voltage_scores)
cnn_metrics = evaluator.calculate_metrics(y_test_can, cnn_predictions, cnn_scores)
lstm_metrics = evaluator.calculate_metrics(y_test_can, lstm_predictions, lstm_scores)

print("Voltage Fingerprinting Metrics:")
for key, value in voltage_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print("\nCNN Metrics:")
for key, value in cnn_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print("\nLSTM Metrics:")
for key, value in lstm_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

In [ ]:
# Visualize results
evaluator.plot_confusion_matrix(y_test_v, voltage_predictions, 
                               title="Voltage Fingerprinting",
                               save_name="demo_voltage_cm.png")

evaluator.plot_confusion_matrix(y_test_can, cnn_predictions,
                               title="CNN Model",
                               save_name="demo_cnn_cm.png")

evaluator.plot_confusion_matrix(y_test_can, lstm_predictions,
                               title="LSTM Model",
                               save_name="demo_lstm_cm.png")

In [ ]:
# ROC Curves
evaluator.plot_roc_curve(y_test_can, cnn_scores,
                        title="CNN - ROC Curve",
                        save_name="demo_cnn_roc.png")

evaluator.plot_roc_curve(y_test_can, lstm_scores,
                        title="LSTM - ROC Curve",
                        save_name="demo_lstm_roc.png")

## 6. Model Comparison

In [ ]:
# Compare all models
comparison_results = {
    'Voltage': voltage_metrics,
    'CNN': cnn_metrics,
    'LSTM': lstm_metrics
}

evaluator.compare_models(
    comparison_results,
    metric_names=['accuracy', 'precision', 'recall', 'f1_score'],
    save_name="demo_comparison.png"
)

## 7. Fusion Layer (Optional)

In [ ]:
from fusion_layer import FusionLayer

# Align data (use minimum length)
min_len = min(len(voltage_scores), len(cnn_scores))

v_scores = voltage_scores[:min_len]
c_scores = cnn_scores[:min_len]
v_conf = 1.0 - v_scores
c_conf = np.abs(c_scores - 0.5) * 2
y_aligned = y_test_can[:min_len]

# Split for fusion training
split = int(0.7 * min_len)

# Train fusion
fusion = FusionLayer(method='weighted_average')
fusion.train(
    v_scores[:split], c_scores[:split],
    v_conf[:split], c_conf[:split],
    y_aligned[:split]
)

# Test fusion
fusion_predictions, fusion_confidences = fusion.predict_batch(
    v_scores[split:], c_scores[split:],
    v_conf[split:], c_conf[split:]
)

# Evaluate
fusion_metrics = evaluator.calculate_metrics(
    y_aligned[split:],
    fusion_predictions,
    fusion_confidences
)

print("Fusion Layer Metrics:")
for key, value in fusion_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")